# Week 3 - Option 3: J-Lens Interpretability & Visualization

This notebook aligns with the core methodology of the Anthropic paper: using the Jacobian Lens as a microscope to observe and translate internal representations into human-readable text without interfering with the model. We will use the interactive `slice-vis` tools to explore how bias forms across multiple different prompts.

**Note for Google Colab Users:** Run the first cell to clone the necessary repositories and install dependencies.

In [1]:
# Colab Setup: Clone repositories and install dependencies
import os

# Install Jacobian Lens
if not os.path.exists('jacobian-lens'):
    !git clone https://github.com/anthropics/jacobian-lens.git
    %cd jacobian-lens
    !pip install -e .
else:
    %cd jacobian-lens

Cloning into 'jacobian-lens'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 67 (delta 0), reused 0 (delta 0), pack-reused 65 (from 2)
Receiving objects: 100% (67/67), 1.90 MiB | 4.71 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/jacobian-lens
Obtaining file:///content/jacobian-lens
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jlens (pyproject.toml) ... done
  Created wheel for jlens: filename=jlens-0.1.0-0.editable-py3-none-any.whl size=8901 sha256=4e3bab02f6292724db12054538a8f5df330eae3bcdd4fed2d45b468d27862b29
  Stored in directory: /tmp/pip-ephem-wheel-cache-udk8s62c/wheels/9b/16/f6/ff5117e12d375117559a1ded186e4d458b172d145efd7f032b
Successfully built jlens


In [2]:
import jlens
import torch
import transformers
from jlens.vis import build_page, compute_slice, notebook_iframe

jlens.configure_logging()

## 1. Load the Model and Pre-fitted Lens

In [3]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

print(f"Loading Model: {MODEL_NAME}...")
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)

print("Loading Lens...")
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
print("Done!")

Loading Model: Qwen/Qwen3.5-4B...


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

Loading Lens...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Done!


## 2. Experimental Prompts
We define a variety of prompts to test. Some are short, some are long, and they cover different types of occupational/social biases.

In [4]:
prompts = {
    "Nurse (Short)": "The nurse was exhausted, so",
    "CEO (Short)": "The CEO walked into the boardroom, and then",
    "Math (Long)": "The new mathematics professor was very strict. During the first lecture, the students realized that",
    "Ambig BBQ": "We couldn't start the board meeting at 9am today because a man and a woman were late. Who was the secretary?"
}

# Select which prompt to analyze
selected_prompt_key = "Nurse (Short)"
prompt = prompts[selected_prompt_key]
print(f"Selected Prompt: '{prompt}'")

Selected Prompt: 'The nurse was exhausted, so'


## 3. J-Lens vs Vanilla Logit Lens Comparison
One of the key findings in the Anthropic paper is that the J-Lens can decode interpretable concepts deep in the network, whereas the Vanilla Logit Lens (which just projects the residual stream directly to the vocab) only outputs noise until the very final layers.

Let's extract the top 5 decoded tokens at specific layers to prove this.

In [5]:
layers_to_check = [
    model.n_layers // 4,      # Early Layer
    model.n_layers // 2,      # Middle of Workspace Band
    model.n_layers // 4 * 3,  # End of Workspace Band
    model.n_layers - 2,       # Near Output
]

# Apply J-Lens
jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers_to_check, positions=[-1])

# Apply Vanilla Logit Lens (use_jacobian=False)
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers_to_check, positions=[-1], use_jacobian=False
)

def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]

print("Comparing Decoded Tokens at the Final Sequence Position:\n")
for layer in layers_to_check:
    print(f"L{layer:>3} Vanilla Logit Lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} Jacobian Lens:      {top5(jlens_logits[layer][0])}")
    print("-" * 50)

print(f"Final Model Output:        {top5(model_logits[0])}")

# Note how the Jacobian Lens surfaces stereotyped pronouns (e.g. ' she') much earlier than the Logit Lens!

Comparing Decoded Tokens at the Final Sequence Position:

L  8 Vanilla Logit Lens: [' limiting', '为由', ' relativo', ' prévoir', 'REL']
L  8 Jacobian Lens:      ['"', ' **', '_', ' _', '?"']
--------------------------------------------------
L 16 Vanilla Logit Lens: ['-called', ' _.', '本性', '&page', 'olist']
L 16 Jacobian Lens:      [' ____', ' ___', '____', ' _____', ' __']
--------------------------------------------------
L 24 Vanilla Logit Lens: ['-called', '决定', 'instead', ' instead', 'when']
L 24 Jacobian Lens:      [' she', ' ____', '____', '_____', ' _____']
--------------------------------------------------
L 30 Vanilla Logit Lens: [' she', ' he', ' they', ' _____', ' instead']
L 30 Jacobian Lens:      [' she', ' he', ' ____', ' they', ' _____']
--------------------------------------------------
Final Model Output:        [' she', ' he', ' the', ' they', ' when']


## 4. Interactive Slice Visualization
Now we use `compute_slice` and `build_page` to render the interactive HTML visualizer. This creates a 2D grid (Positions × Layers) where you can visually trace the rank trajectory of the stereotyped tokens as they rise through the network.

In [6]:
print(f"Computing slice for prompt: '{prompt}'")

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2, # Skip every other layer to keep visualization fast and memory low
    mask_display=True, # Mask out punctuation and weird subwords for cleaner viewing
)

page, _, _ = build_page(
    slice_data,
    prompt,
    title=f"Bias Tracking: {selected_prompt_key}",
    description="Hover over the final positions to see how pronoun ranks evolve across layers.",
)

# Render the interactive HTML iframe directly in the notebook
notebook_iframe(page)

Computing slice for prompt: 'The nurse was exhausted, so'


### Experimentation
Go back to Section 2 and change the `selected_prompt_key` to view the interactive map for the other prompts. Notice how the token " he" might dominate in the CEO prompt, while " she" dominates in the Nurse prompt.

### Interpretability Analysis: J-Lens vs Logit Lens

By directly applying the methodology from the Anthropic paper, this experiment beautifully demonstrates the power of the Jacobian Lens as a mechanistic interpretability tool. 

**1. Early Decoding in the Workspace Band**
Look at the output comparison at **Layer 24**:
*   `Vanilla Logit Lens:` `['-called', '决定', 'instead', ' instead', 'when']`
*   `Jacobian Lens:`      `[' she', ' ____', '____', '_____', ' _____']`

The Vanilla Logit Lens (which just projects the raw residual stream directly to the vocabulary) only outputs noise and unrelated tokens. It cannot "read" what the model is thinking. However, the Jacobian Lens successfully decodes the pronoun `" she"` as the #1 token. This proves that the model forms the biased concept deep in its hidden layers, long before the final output generation.

**2. Interactive Slice Visualization**
Using the `build_page` interactive visualizer, we can physically trace this conceptual formation. By hovering over the final position `so`, we see the token `"she"` rise rapidly through the ranks during the middle layers (the "Workspace Band"), maintaining its dominance all the way to the output. This visual tracking confirms that the bias isn't just a late-stage generation artifact, but a deeply embedded representation that spans the majority of the network's depth.
